# B2-022-probabilistic-latent-models — Practice p05 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** gaussian-reparameterization

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

**Answer: C.**
Sampling $z\sim N(\mu,\operatorname{diag}e^{\ell})$ with a sampler is not a differentiable function of $(\mu,\ell)$.
Writing $z=\mu+\exp(\tfrac12\ell)\odot\varepsilon$ with parameter-free $\varepsilon\sim N(0,I)$ keeps the distribution of $z$ the same while making $z$ (and hence the Monte Carlo reconstruction term) a differentiable function of $\mu$, $\ell$, and the encoder weights: $\partial z/\partial\mu=1$ and $\partial z/\partial\ell=\tfrac12e^{\ell/2}\varepsilon$.

Rejected options:

- **A.** $\varepsilon$ is fixed noise drawn independently of all parameters; it is never learned or updated.
- **B.** The closed-form KL is already an explicit smooth function of $\mu$ and $\ell$, so it needs no trick to be differentiable.
- **D.** $\log p(x)$ remains intractable; the trick only gives low-variance gradients of the ELBO, which is still the objective.
- **E.** The decoder is a neural network and is differentiable in its own weights regardless; the trick concerns gradients reaching the *encoder* through the sample.

In [ ]:
import torch

mu = torch.tensor([[0.3, -0.2]], requires_grad=True)
logvar = torch.tensor([[0.0, -1.0]], requires_grad=True)
eps = torch.randn(1, 2, generator=torch.Generator().manual_seed(20260927))
z = mu + torch.exp(0.5 * logvar) * eps
(z ** 2).sum().backward()
with torch.no_grad():
    # torch.normal(mean, std) draws one Gaussian sample per entry with the
    # given mean and standard deviation tensors
    blocked = torch.normal(mu, torch.exp(0.5 * logvar))   # direct sampling: no graph to mu/logvar
ANSWER = "C"
print(mu.grad, logvar.grad, blocked.requires_grad)

### Answer check

In [ ]:
# Checks: the reparameterization answers against pinned values
assert torch.allclose(mu.grad, 2 * z.detach(), atol=1e-6, rtol=1e-6)
assert torch.allclose(logvar.grad, 2 * z.detach() * 0.5 * torch.exp(0.5 * logvar.detach()) * eps, atol=1e-6, rtol=1e-6)
assert eps.grad is None and not eps.requires_grad
assert blocked.requires_grad is False
assert ANSWER == "C"